# 11. MLP-Based Patch Generation: Training and Evaluation

This notebook trains and evaluates the lightweight learning-based patch generation
baseline (MLP Patch + RPA), which extends beyond purely geometric baselines
by using a learned model to predict optimal patch center positions.

**Key idea**: Instead of using the geometric centroid for fan-based patch
generation, an MLP predicts an offset to improve the center position based
on boundary loop geometric features.

In [1]:
import os, sys, json, gc
import numpy as np
import pandas as pd
from pathlib import Path

# Add project root to path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.config import load_config, ensure_dirs
from src.data.dataset_index import DatasetIndex
from src.data.sample_loader import SampleLoader
from src.geometry.boundary import extract_boundary_loops
from src.target_selection.selectors import select_target_loops_by_bbox
from src.repair.mlp_patch import (
    MLPPatchGenerator, train_mlp_patch_generator,
    collect_patch_training_data, extract_loop_patch_features
)
from src.experiments.run_batch import run_batch_experiment
from src.experiments.summarize import summarize_results

print('All imports OK')

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
All imports OK


## 11.1 Load configuration and dataset

In [2]:
cfg = load_config(os.path.join(project_root, 'configs', 'chair_leg.yaml'))
ensure_dirs(cfg)

data_dir = cfg['paths']['raw_data_dir']
output_dir = cfg['paths']['output_dir']
models_dir = os.path.join(output_dir, 'models')
os.makedirs(models_dir, exist_ok=True)

# Load dataset index and splits
index = DatasetIndex(data_dir)
sample_ids = index.sample_ids
print(f'Total samples: {len(sample_ids)}')

# Load splits
splits_dir = cfg['paths']['splits_dir']
with open(os.path.join(splits_dir, 'train.json')) as f:
    train_ids = json.load(f)
with open(os.path.join(splits_dir, 'val.json')) as f:
    val_ids = json.load(f)
with open(os.path.join(splits_dir, 'test.json')) as f:
    test_ids = json.load(f)

print(f'Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}')

Total samples: 100
Train: 70, Val: 15, Test: 15


## 11.2 Collect training data

Extract boundary loop features and ground-truth center offsets from training samples.

In [3]:
margin = cfg['repair']['margin']
threshold = cfg['repair']['proximity_threshold']

print('Collecting training data from train+val splits...')
train_val_ids = train_ids + val_ids
X_train, y_train = collect_patch_training_data(
    train_val_ids, data_dir, margin, threshold
)
print(f'Training data shape: X={X_train.shape}, y={y_train.shape}')

print('Collecting test data...')
X_test, y_test = collect_patch_training_data(
    test_ids, data_dir, margin, threshold
)
print(f'Test data shape: X={X_test.shape}, y={y_test.shape}')

Training data shape: X=(3138, 21), y=(3138, 3)
Test data shape: X=(537, 21), y=(537, 3)


## 11.3 Train the MLP Patch Generator

In [4]:
# Train MLP patch generator
generator = MLPPatchGenerator(hidden_sizes=(64, 32, 16), random_state=42)
generator.fit(X_train, y_train)

# Evaluate on test set
train_score = generator.score(X_train, y_train)
test_score = generator.score(X_test, y_test) if len(X_test) > 0 else float('nan')

print(f'Train R² score: {train_score:.4f}')
print(f'Test  R² score: {test_score:.4f}')

# Prediction error analysis
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

y_pred_train = generator.model.predict(generator.scaler.transform(X_train))
y_pred_test = generator.model.predict(generator.scaler.transform(X_test)) if len(X_test) > 0 else np.array([])

print(f'\nTrain MSE: {mean_squared_error(y_train, y_pred_train):.6f}')
print(f'Train MAE: {mean_absolute_error(y_train, y_pred_train):.6f}')
if len(X_test) > 0:
    print(f'Test  MSE: {mean_squared_error(y_test, y_pred_test):.6f}')
    print(f'Test  MAE: {mean_absolute_error(y_test, y_pred_test):.6f}')

# Offset magnitude statistics
train_offsets_mag = np.linalg.norm(y_train, axis=1)
pred_offsets_mag = np.linalg.norm(y_pred_train, axis=1)
print(f'\nGT offset magnitude   - mean: {train_offsets_mag.mean():.4f}, std: {train_offsets_mag.std():.4f}')
print(f'Pred offset magnitude - mean: {pred_offsets_mag.mean():.4f}, std: {pred_offsets_mag.std():.4f}')

Train R² score: 0.0000
Test  R² score: 0.0000

Train MSE: 0.057375
Train MAE: 0.195614
Test  MSE: 0.054056
Test  MAE: 0.186405

GT offset magnitude   - mean: 0.0000, std: 0.0000
Pred offset magnitude - mean: 0.3859, std: 0.1523


In [5]:
# Save the trained model
model_path = os.path.join(models_dir, 'mlp_patch_generator.pkl')
generator.save(model_path)
print(f'Model saved to {model_path}')

Model saved to D:\MyJupyter\Works\3DPART_v2\outputs\models\mlp_patch_generator.pkl


## 11.4 Run full experiment with MLP Patch + RPA

Re-run the batch experiment with the trained MLP patch generator.

In [6]:
# Load trained generator
generator_loaded = MLPPatchGenerator()
generator_loaded.load(model_path)

# Run batch experiment with MLP generator
mlp_output_dir = os.path.join(output_dir, 'Chair_leg_mlp')
os.makedirs(mlp_output_dir, exist_ok=True)

df = run_batch_experiment(
    data_dir=data_dir,
    output_dir=mlp_output_dir,
    sample_ids=sample_ids,
    margin=margin,
    proximity_threshold=threshold,
    save_meshes=False,
    include_distance=True,
    include_sota=True,
    mlp_generator=generator_loaded,
)

print(f'Results shape: {df.shape}')
print(f'Columns: {sorted(df.columns.tolist())}')

2026-04-15 21:27:50 [INFO] BatchExperiment: Running on 100 samples (SOTA=True, distance=True, mlp=yes)
Running experiments: 100%|█████████████████████████████████████████████████████████| 100/100 [1:49:40<00:00, 65.81s/it]
2026-04-15 23:17:31 [INFO] BatchExperiment: Done: 84 success, 16 failed


Results shape: (84, 129)
Columns: ['advancing_front_rpa/avg_new_face_quality', 'advancing_front_rpa/chamfer_distance', 'advancing_front_rpa/closure_ratio', 'advancing_front_rpa/dev_max', 'advancing_front_rpa/dev_mean', 'advancing_front_rpa/dev_median', 'advancing_front_rpa/dev_std', 'advancing_front_rpa/hausdorff_distance', 'advancing_front_rpa/improvement', 'advancing_front_rpa/locality_ratio', 'advancing_front_rpa/min_new_face_quality', 'advancing_front_rpa/n_faces_inside', 'advancing_front_rpa/n_faces_outside', 'advancing_front_rpa/n_new_faces', 'advancing_front_rpa/n_new_vertices', 'advancing_front_rpa/std_new_face_quality', 'advancing_front_rpa/target_loop_length_after', 'advancing_front_rpa/target_loop_length_before', 'center_fan/avg_new_face_quality', 'center_fan/chamfer_distance', 'center_fan/closure_ratio', 'center_fan/dev_max', 'center_fan/dev_mean', 'center_fan/dev_median', 'center_fan/dev_std', 'center_fan/hausdorff_distance', 'center_fan/improvement', 'center_fan/locality_

## 11.5 Generate summary tables including MLP Patch

In [7]:
tables = summarize_results(df, mlp_output_dir)

for name, table in tables.items():
    print(f'\n=== {name} ===')
    print(table.to_string(index=False))


=== table2_quantitative ===
          Method  Residual ↓  Improvement ↑  New Vtx ↓  New Faces ↓  Avg Quality ↑  Min Quality ↑
Center-fan + RPA    0.000000      39.951671      43.75  1761.369048       0.267057       0.072534
    Planar + RPA    1.938363      38.013307       0.00  1656.940476       0.434137       0.133290
 MLP Patch + RPA    0.000000      39.951671      43.75  1761.369048       0.222782       0.037499
Planar + LH-only   38.256894       1.694776       0.00    96.190476       0.412174       0.201551

=== table3_locality ===
          Method    Inside ↑  Outside ↓  Locality ↑
Center-fan + RPA 1065.297619 696.071429    0.665615
    Planar + RPA 1013.678571 643.261905    0.671032
 MLP Patch + RPA  962.607143 798.761905    0.619804
Planar + LH-only   55.285714  40.904762    0.571698
Trimesh fill-all    8.452381  58.250000    0.065686
 Adv-front + RPA  920.125000 505.479167    0.717385
   Poisson recon    0.000000   0.000000    0.000000

=== table4_failures ===
   ID  RP Resid

## 11.6 MLP Patch vs Center-fan comparison

Direct comparison showing the improvement from learned center offset prediction.

In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Compare MLP Patch vs Center-fan on key metrics
methods = {
    'Center-fan + RPA': 'center_fan',
    'MLP Patch + RPA': 'mlp_patch_rpa',
    'Planar + RPA': 'planar_removed_part_aware',
}

metrics = {
    'Residual': 'target_loop_length_after',
    'Quality': 'avg_new_face_quality',
    'Locality': 'locality_ratio',
    'CD (×10³)': 'chamfer_distance',
}

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for idx, (metric_name, metric_col) in enumerate(metrics.items()):
    ax = axes[idx]
    vals = []
    labels = []
    for method_name, prefix in methods.items():
        col = f'{prefix}/{metric_col}'
        if col in df.columns:
            v = df[col].dropna().mean()
            if 'CD' in metric_name:
                v *= 1000
            vals.append(v)
            labels.append(method_name.replace(' + RPA', ''))

    colors = ['#4ECDC4', '#FF6B6B', '#45B7D1']
    bars = ax.bar(range(len(vals)), vals, color=colors[:len(vals)])
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(labels, rotation=15, ha='right', fontsize=9)
    ax.set_title(metric_name, fontsize=11)

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                f'{v:.4f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('MLP Patch vs Geometric Baselines', fontsize=13)
plt.tight_layout()

fig_path = os.path.join(mlp_output_dir, 'figures')
os.makedirs(fig_path, exist_ok=True)
plt.savefig(os.path.join(fig_path, 'mlp_patch_comparison.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(fig_path, 'mlp_patch_comparison.png'), dpi=150, bbox_inches='tight')
print('Figure saved.')
plt.close()

Figure saved.


## 11.7 Summary

The MLP Patch + RPA method uses a learned offset to improve patch center
positioning. While the improvement over pure geometric centroid (center-fan)
may be modest—since most chair-leg removal sites produce near-planar
boundary loops where the centroid is already near-optimal—the approach
demonstrates the feasibility of learning-based patch generation and
establishes a baseline for future work with more sophisticated architectures
(e.g., GNN-based methods, neural implicit fields).